# Car Price NLP Block
## Natural Language Processing — Feature Extraction & Explanation

**Goal:** This block serves two purposes:
1. **Input:** Parse a natural language car description into structured features that feed the ML price prediction model
2. **Output:** Generate a plain-language explanation of the predicted price, including the most significant factors

**Model:** OpenAI `gpt-4o-mini` (primary), compared against `gpt-4o` in Iteration 3

**Integration:**
```
Text description → [NLP: extraction] → structured features → ML → predicted price
                                                                        ↓
                              [NLP: explanation] ← feature importances ←
                                        ↓
                              Plain-language explanation shown to user
```

## Project Setup
### Libraries and Settings

In [3]:
!pip install -q openai python-dotenv pydantic

In [4]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from typing import Optional
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env file
client = OpenAI()

print('OpenAI client ready')

OpenAI client ready


## 1. Data — Test Cases and ML Model

We define 10 hand-crafted car descriptions with known correct feature values to evaluate extraction quality across iterations.

In [5]:
# 10 test cases: natural language description + expected extracted features
test_cases = [
    {
        "description": "2019 BMW 3 Series, 45,000 miles, gasoline, automatic, no accidents, clean title",
        "expected": {"brand": "BMW", "model_year": 2019, "milage": 45000, "fuel_type": "Gasoline", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}
    },
    {
        "description": "Selling my 2015 Toyota Camry with 87k miles on it. Manual transmission, runs on gas. Had one minor fender bender but still has a clean title.",
        "expected": {"brand": "Toyota", "model_year": 2015, "milage": 87000, "fuel_type": "Gasoline", "transmission": "Manual", "has_accident": 1, "clean_title_flag": 1}
    },
    {
        "description": "2022 Tesla Model 3, fully electric, 28000 miles, automatic, never been in an accident, clean title.",
        "expected": {"brand": "Tesla", "model_year": 2022, "milage": 28000, "fuel_type": "Electric", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}
    },
    {
        "description": "Ford F-150 from 2017, diesel engine, 120,000 miles, automatic gearbox, been in two accidents, salvage title.",
        "expected": {"brand": "Ford", "model_year": 2017, "milage": 120000, "fuel_type": "Diesel", "transmission": "Automatic", "has_accident": 1, "clean_title_flag": 0}
    },
    {
        "description": "I'm looking to price my 2020 Honda Civic. It's a hybrid with 33,000 miles, automatic, accident-free with clean title.",
        "expected": {"brand": "Honda", "model_year": 2020, "milage": 33000, "fuel_type": "Hybrid", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}
    },
    {
        "description": "2018 Mercedes-Benz C300, 62000 miles, petrol, automatic, no accidents ever, title is clean.",
        "expected": {"brand": "Mercedes-Benz", "model_year": 2018, "milage": 62000, "fuel_type": "Gasoline", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}
    },
    {
        "description": "Chevrolet Silverado 2016, gas powered, manual transmission, 155000 miles, had 3 accidents, no clean title.",
        "expected": {"brand": "Chevrolet", "model_year": 2016, "milage": 155000, "fuel_type": "Gasoline", "transmission": "Manual", "has_accident": 1, "clean_title_flag": 0}
    },
    {
        "description": "Just bought a Porsche 911 back in 2021, 11000 miles, gasoline, PDK automatic. Never had any accident. Clean title.",
        "expected": {"brand": "Porsche", "model_year": 2021, "milage": 11000, "fuel_type": "Gasoline", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}
    },
    {
        "description": "2014 Jeep Wrangler, 4WD, petrol engine, manual, 98000 miles on the clock. One accident on record, still clean title.",
        "expected": {"brand": "Jeep", "model_year": 2014, "milage": 98000, "fuel_type": "Gasoline", "transmission": "Manual", "has_accident": 1, "clean_title_flag": 1}
    },
    {
        "description": "Selling a 2023 Audi A4 with only 5000 miles. Diesel engine, automatic. Zero accidents. Clean title of course.",
        "expected": {"brand": "Audi", "model_year": 2023, "milage": 5000, "fuel_type": "Diesel", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}
    }
]

print(f'Test cases loaded: {len(test_cases)}')

Test cases loaded: 10


In [6]:
# Load the ML model to use for price prediction in integration tests
with open('../01_ML_Numeric/car_price_model.pkl', 'rb') as f:
    model_payload = pickle.load(f)

ml_model  = model_payload['model']
features  = model_payload['features']
le        = model_payload['label_encoders']

print('ML model loaded. Features:', features)

# Feature importances from ML model (used in explanation prompt)
importances = dict(zip(features, ml_model.feature_importances_))
top_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:5]
print('Top 5 features by importance:')
for feat, imp in top_features:
    print(f'  {feat}: {imp:.4f}')

ML model loaded. Features: ['model_year', 'milage', 'car_age', 'age_times_milage', 'fuel_type_enc', 'transmission_enc', 'brand_enc', 'has_accident', 'clean_title_flag', 'engine_hp', 'condition_score']
Top 5 features by importance:
  age_times_milage: 0.4739
  engine_hp: 0.3126
  brand_enc: 0.0778
  milage: 0.0655
  fuel_type_enc: 0.0223


c:\Users\nicof\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nicof\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nicof\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.2. This might lead

In [7]:
# Helper: build ML input from extracted features and predict price
def predict_price(extracted):
    brand        = extracted.get('brand', 'Other')
    model_year   = extracted.get('model_year', 2015)
    milage       = extracted.get('milage', 60000)
    fuel_type    = extracted.get('fuel_type', 'Gasoline')
    transmission = extracted.get('transmission', 'Automatic')
    has_accident = extracted.get('has_accident', 0)
    clean_title  = extracted.get('clean_title_flag', 1)
    engine_hp    = extracted.get('engine_hp', 180)
    condition_score = extracted.get('condition_score', 0)

    car_age          = 2024 - model_year
    age_times_milage = car_age * milage

    fuel_enc  = le['fuel_type'].transform([fuel_type])[0]  if le.get('fuel_type')  and fuel_type  in le['fuel_type'].classes_  else 0
    trans_enc = le['transmission'].transform([transmission])[0] if le.get('transmission') and transmission in le['transmission'].classes_ else 0
    brand_grp = brand if brand in le['brand'].classes_ else 'Other'
    brand_enc = le['brand'].transform([brand_grp])[0] if le.get('brand') else 0

    row = {
        'model_year': model_year, 'milage': milage, 'car_age': car_age,
        'age_times_milage': age_times_milage, 'fuel_type_enc': fuel_enc,
        'transmission_enc': trans_enc, 'brand_enc': brand_enc,
        'has_accident': has_accident, 'clean_title_flag': clean_title,
        'engine_hp': engine_hp, 'condition_score': condition_score
    }
    X = pd.DataFrame([{f: row.get(f, 0) for f in features}])
    return round(ml_model.predict(X)[0], 0)

### Evaluation Helper

Measures extraction quality by comparing predicted fields to expected values for each test case.

In [8]:
def evaluate_extraction(extracted_list, test_cases):
    fields = ['brand', 'model_year', 'milage', 'fuel_type', 'transmission', 'has_accident', 'clean_title_flag']
    results = []
    for extracted, case in zip(extracted_list, test_cases):
        expected = case['expected']
        correct = sum(1 for f in fields if str(extracted.get(f, '')).lower() == str(expected.get(f, '')).lower())
        results.append({'correct': correct, 'total': len(fields)})
    avg = np.mean([r['correct'] / r['total'] for r in results])
    print(f'Field accuracy: {avg:.2%} ({sum(r["correct"] for r in results)}/{sum(r["total"] for r in results)} fields correct)')
    return avg

---
## Iteration 1 — Zero-Shot Prompt (Simple)

**Objective:** Establish a baseline using a minimal zero-shot prompt for both feature extraction and explanation. No structured output format enforced.

**Model:** `gpt-4o-mini`

**Approach:** Single prompt asking for JSON extraction; separate prompt for explanation.

In [9]:
def extract_features_iter1(description):
    prompt = f"""
Extract car details from this description as JSON.
Return only a JSON object with these keys: brand, model_year, milage, fuel_type, transmission, has_accident (0 or 1), clean_title_flag (0 or 1).

Description: {description}
"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    text = response.choices[0].message.content.strip()
    # Strip markdown code blocks if present
    text = text.replace('```json', '').replace('```', '').strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {}

# Test on first case
sample = test_cases[0]
result = extract_features_iter1(sample['description'])
print('Description:', sample['description'])
print('Extracted: ', result)
print('Expected:  ', sample['expected'])

Description: 2019 BMW 3 Series, 45,000 miles, gasoline, automatic, no accidents, clean title
Extracted:  {'brand': 'BMW', 'model_year': 2019, 'milage': 45000, 'fuel_type': 'gasoline', 'transmission': 'automatic', 'has_accident': 0, 'clean_title_flag': 1}
Expected:   {'brand': 'BMW', 'model_year': 2019, 'milage': 45000, 'fuel_type': 'Gasoline', 'transmission': 'Automatic', 'has_accident': 0, 'clean_title_flag': 1}


In [10]:
# Evaluate Iter 1 extraction on all test cases
extracted_iter1 = [extract_features_iter1(tc['description']) for tc in test_cases]
print('=== Iteration 1 — Feature Extraction ===')
acc_iter1 = evaluate_extraction(extracted_iter1, test_cases)

=== Iteration 1 — Feature Extraction ===
Field accuracy: 92.86% (65/70 fields correct)


In [11]:
def explain_price_iter1(extracted, predicted_price):
    prompt = f"""
A used car price prediction model estimated the price of this car at ${predicted_price:,.0f}.
Car details: {json.dumps(extracted, indent=2)}

Write a short explanation (3-4 sentences) of why this price was estimated and what the most important factors are.
"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

# Demo explanation on first case
sample_extracted = extracted_iter1[0]
sample_price = predict_price(sample_extracted)
explanation = explain_price_iter1(sample_extracted, sample_price)
print(f'Predicted price: ${sample_price:,.0f}')
print(f'\nExplanation (Iter 1):')
print(explanation)

Predicted price: $43,783

Explanation (Iter 1):
The estimated price of $43,783 for the 2019 BMW is influenced by several key factors, including its relatively low mileage of 45,000 miles, which suggests less wear and tear, and its automatic transmission, which is often preferred by buyers for convenience. Additionally, the car's clean title and absence of accidents enhance its value, as these factors indicate a well-maintained vehicle with no significant damage history. The brand reputation of BMW also plays a crucial role, as luxury vehicles typically retain higher resale values compared to non-luxury brands. Overall, the combination of these elements contributes to the higher estimated price.


---
## Iteration 2 — Structured Output with Pydantic

**Objective:** Replace free-form JSON parsing with Pydantic-enforced structured output (same approach as Week 8 exercise). This eliminates parsing errors and enforces correct data types. Explanation prompt is also improved with feature importances from the ML model.

**Model:** `gpt-4o-mini`

**Key changes vs Iteration 1:** Pydantic schema enforces output structure; explanation prompt includes top feature importances.

In [12]:
# Pydantic schema for structured extraction — same pattern as Week 8
class CarFeatures(BaseModel):
    brand: str = Field(description='Car manufacturer, e.g. BMW, Toyota, Ford')
    model_year: int = Field(description='4-digit year of manufacture')
    milage: int = Field(description='Mileage in miles (integer)')
    fuel_type: str = Field(description='One of: Gasoline, Diesel, Electric, Hybrid')
    transmission: str = Field(description='One of: Automatic, Manual')
    has_accident: int = Field(description='1 if any accident history, 0 if none')
    clean_title_flag: int = Field(description='1 if clean title, 0 if salvage/rebuilt')


def extract_features_iter2(description):
    prompt = f'Extract the car features from this description: {description}'
    response = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format=CarFeatures,
        temperature=0
    )
    return response.choices[0].message.parsed.model_dump()

# Test
sample = test_cases[0]
result = extract_features_iter2(sample['description'])
print('Extracted:', result)
print('Expected: ', sample['expected'])

Extracted: {'brand': 'BMW', 'model_year': 2019, 'milage': 45000, 'fuel_type': 'Gasoline', 'transmission': 'Automatic', 'has_accident': 0, 'clean_title_flag': 1}
Expected:  {'brand': 'BMW', 'model_year': 2019, 'milage': 45000, 'fuel_type': 'Gasoline', 'transmission': 'Automatic', 'has_accident': 0, 'clean_title_flag': 1}


In [13]:
# Evaluate Iter 2 extraction
extracted_iter2 = [extract_features_iter2(tc['description']) for tc in test_cases]
print('=== Iteration 2 — Feature Extraction ===')
acc_iter2 = evaluate_extraction(extracted_iter2, test_cases)

=== Iteration 2 — Feature Extraction ===
Field accuracy: 100.00% (70/70 fields correct)


In [14]:
def explain_price_iter2(extracted, predicted_price, top_features):
    feature_str = '\n'.join([f'  - {f}: importance {imp:.3f}' for f, imp in top_features])
    prompt = f"""
You are a car pricing expert. A machine learning model predicted the price of a used car.

Predicted price: ${predicted_price:,.0f}

Car details:
{json.dumps(extracted, indent=2)}

The model's top 5 most important features (by weight) were:
{feature_str}

Write a clear explanation (3-5 sentences) of the predicted price. Focus on the most impactful features.
Mention the brand, age, mileage and condition. Be specific about which factors push the price up or down.
"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

# Demo explanation
sample_extracted = extracted_iter2[0]
sample_price = predict_price(sample_extracted)
explanation = explain_price_iter2(sample_extracted, sample_price, top_features)
print(f'Predicted price: ${sample_price:,.0f}')
print(f'\nExplanation (Iter 2):')
print(explanation)

Predicted price: $21,383

Explanation (Iter 2):
The predicted price of the used 2019 BMW, set at $21,383, is significantly influenced by its age and mileage. At just four years old, the car benefits from a relatively low age, which typically helps maintain a higher resale value. Additionally, with only 45,000 miles driven, the mileage is below average for its age, further enhancing its desirability and price. The car's clean title and lack of accidents also contribute positively to its valuation, as these factors indicate better overall condition and reliability. Overall, the combination of a reputable brand, low mileage, and good condition supports the higher predicted price.


---
## Iteration 3 — System Prompt + Few-Shot + Model Comparison (gpt-4o-mini vs gpt-4o)

**Objective:** Add a system prompt providing context, and few-shot examples for harder edge cases. Compare output quality between `gpt-4o-mini` (cheap) and `gpt-4o` (powerful) on a sample of 3 test cases.

**Key changes vs Iteration 2:** System message with domain context; 2 few-shot examples in prompt; model comparison.

In [15]:
SYSTEM_PROMPT = """
You are a car data extraction specialist. Your task is to extract structured information from
natural language car descriptions. Be precise:
- Normalise fuel types to: Gasoline, Diesel, Electric, Hybrid
- Normalise transmission to: Automatic, Manual
- Any mention of accident, collision, fender bender → has_accident = 1
- Salvage/rebuilt/non-clean title → clean_title_flag = 0
"""

FEW_SHOT_EXAMPLES = """
Example 1:
Description: "2017 Audi Q5, 73k miles, petrol, PDK gearbox, clean accident history, original title"
Output: {"brand": "Audi", "model_year": 2017, "milage": 73000, "fuel_type": "Gasoline", "transmission": "Automatic", "has_accident": 0, "clean_title_flag": 1}

Example 2:
Description: "Used 2013 Dodge Ram, diesel, manual, 210,000 miles on it, been in one accident, salvage title."
Output: {"brand": "Dodge", "model_year": 2013, "milage": 210000, "fuel_type": "Diesel", "transmission": "Manual", "has_accident": 1, "clean_title_flag": 0}
"""

def extract_features_iter3(description, model='gpt-4o-mini'):
    prompt = f"""{FEW_SHOT_EXAMPLES}
Now extract from this description:
{description}"""
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt}
        ],
        response_format=CarFeatures,
        temperature=0
    )
    return response.choices[0].message.parsed.model_dump()

# Evaluate gpt-4o-mini
extracted_iter3_mini = [extract_features_iter3(tc['description'], model='gpt-4o-mini') for tc in test_cases]
print('=== Iteration 3 — gpt-4o-mini ===')
acc_iter3_mini = evaluate_extraction(extracted_iter3_mini, test_cases)

=== Iteration 3 — gpt-4o-mini ===
Field accuracy: 100.00% (70/70 fields correct)


In [16]:
# Compare with gpt-4o on 3 hardest cases (indices 1, 3, 8 — contain edge cases)
hard_cases = [test_cases[1], test_cases[3], test_cases[8]]
extracted_gpt4o = [extract_features_iter3(tc['description'], model='gpt-4o') for tc in hard_cases]
extracted_gpt4o_mini = [extracted_iter3_mini[1], extracted_iter3_mini[3], extracted_iter3_mini[8]]

print('=== Model Comparison on 3 Edge Cases ===')
print('gpt-4o-mini:')
acc_mini = evaluate_extraction(extracted_gpt4o_mini, hard_cases)
print('gpt-4o:')
acc_gpt4o = evaluate_extraction(extracted_gpt4o, hard_cases)

print(f'\ngpt-4o-mini accuracy: {acc_mini:.2%}')
print(f'gpt-4o accuracy:      {acc_gpt4o:.2%}')

=== Model Comparison on 3 Edge Cases ===
gpt-4o-mini:
Field accuracy: 100.00% (21/21 fields correct)
gpt-4o:
Field accuracy: 100.00% (21/21 fields correct)

gpt-4o-mini accuracy: 100.00%
gpt-4o accuracy:      100.00%


In [17]:
def explain_price_iter3(extracted, predicted_price, top_features, model='gpt-4o-mini'):
    feature_str = '\n'.join([f'  - {f} (importance: {imp:.3f})' for f, imp in top_features])
    user_prompt = f"""
Predicted price: ${predicted_price:,.0f}

Car details:
{json.dumps(extracted, indent=2)}

The machine learning model's top 5 most important features are:
{feature_str}

Provide a structured explanation with:
1. One sentence summarising the estimated price
2. Two or three bullet points identifying the most significant price drivers (positive and negative)
3. One sentence noting any limitations or uncertainty
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You are a car valuation expert. Explain used car price estimates clearly and concisely for a non-technical audience.'},
            {'role': 'user',   'content': user_prompt}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

# Demo explanation — Iter 3
sample_extracted = extracted_iter3_mini[0]
sample_price = predict_price(sample_extracted)
explanation = explain_price_iter3(sample_extracted, sample_price, top_features)
print(f'Predicted price: ${sample_price:,.0f}')
print(f'\nExplanation (Iter 3):')
print(explanation)

Predicted price: $21,383

Explanation (Iter 3):
1. The estimated price for the 2019 BMW with 45,000 miles is $21,383.

2. Significant price drivers:
   - **Age and Mileage**: The combination of the car's age and mileage is the most influential factor, with lower mileage generally leading to a higher value.
   - **Engine Power**: The horsepower of the engine also plays a significant role, as more powerful engines typically enhance a car's desirability and price.
   - **Brand Reputation**: BMW's brand recognition contributes positively to the car's value, reflecting its reputation for quality and performance.

3. It's important to note that this estimate may vary based on local market conditions, additional features, and the car's overall condition.


## 4. Comparison Summary

In [18]:
print('=== Feature Extraction Accuracy (all 10 test cases) ===')
print(f'Iter 1 – Zero-shot simple prompt:              {acc_iter1:.2%}')
print(f'Iter 2 – Pydantic structured output:           {acc_iter2:.2%}')
print(f'Iter 3 – System prompt + few-shot (mini):      {acc_iter3_mini:.2%}')
print()
print('=== Explanation Quality (qualitative, 1=poor 5=excellent) ===')
print('Iter 1 – Basic prompt, no context:              [fill after review]')
print('Iter 2 – Feature importances included:          [fill after review]')
print('Iter 3 – System prompt + structured output:     [fill after review]')

=== Feature Extraction Accuracy (all 10 test cases) ===
Iter 1 – Zero-shot simple prompt:              92.86%
Iter 2 – Pydantic structured output:           100.00%
Iter 3 – System prompt + few-shot (mini):      100.00%

=== Explanation Quality (qualitative, 1=poor 5=excellent) ===
Iter 1 – Basic prompt, no context:              [fill after review]
Iter 2 – Feature importances included:          [fill after review]
Iter 3 – System prompt + structured output:     [fill after review]


In [19]:
# Side-by-side explanation comparison on the same car
desc = test_cases[0]['description']
ext  = extract_features_iter3(desc)
price = predict_price(ext)

print('=== Explanation Comparison (same car, same price) ===')
print(f'Description: {desc}')
print(f'Predicted price: ${price:,.0f}')
print()
print('--- Iter 1 ---')
print(explain_price_iter1(ext, price))
print()
print('--- Iter 2 ---')
print(explain_price_iter2(ext, price, top_features))
print()
print('--- Iter 3 ---')
print(explain_price_iter3(ext, price, top_features))

=== Explanation Comparison (same car, same price) ===
Description: 2019 BMW 3 Series, 45,000 miles, gasoline, automatic, no accidents, clean title
Predicted price: $21,383

--- Iter 1 ---
The estimated price of $21,383 for the 2019 BMW is influenced by several key factors. Firstly, the relatively low mileage of 45,000 miles suggests that the car has been used less than average, which typically enhances its value. Additionally, the car's fuel type (gasoline) and automatic transmission are popular features that appeal to a broad range of buyers. The absence of accidents and the presence of a clean title further contribute to a higher valuation, as these factors indicate the car's overall condition and reliability.

--- Iter 2 ---
The predicted price of the used BMW, set at $21,383, reflects several key factors influencing its value. The car's relatively low mileage of 45,000 miles and its status as a 2019 model contribute positively to its price, as these elements suggest less wear and t

## 5. Integration Test — Full Pipeline

Demonstrates the complete flow from a user's text description to a price estimate and explanation.

In [20]:
def full_pipeline(user_description, condition_score=0):
    """
    Full NLP → ML → NLP pipeline.
    condition_score: output from CV block (0=minor, 1=moderate, 2=severe damage)
    """
    print('Input:', user_description)
    print()

    # Step 1: Extract features from text (best prompt — Iter 3)
    extracted = extract_features_iter3(user_description)
    extracted['condition_score'] = condition_score
    print('Extracted features:', extracted)
    print()

    # Step 2: Predict price with ML model
    price = predict_price(extracted)
    print(f'Predicted price: ${price:,.0f}')
    print()

    # Step 3: Generate explanation (best prompt — Iter 3)
    explanation = explain_price_iter3(extracted, price, top_features)
    print('Explanation:')
    print(explanation)
    print()

    return {'extracted': extracted, 'price': price, 'explanation': explanation}


# Test with two example inputs
result1 = full_pipeline(
    "My 2019 BMW 3 Series has 45,000 miles, gasoline, automatic, never had an accident, clean title.",
    condition_score=0  # no damage from CV block
)

print('─' * 60)

result2 = full_pipeline(
    "Selling a 2015 Toyota Camry. Around 90k miles, runs on gas, automatic gearbox. Had a minor fender bender last year but still clean title.",
    condition_score=1  # minor damage from CV block
)

Input: My 2019 BMW 3 Series has 45,000 miles, gasoline, automatic, never had an accident, clean title.

Extracted features: {'brand': 'BMW', 'model_year': 2019, 'milage': 45000, 'fuel_type': 'Gasoline', 'transmission': 'Automatic', 'has_accident': 0, 'clean_title_flag': 1, 'condition_score': 0}

Predicted price: $21,383

Explanation:
1. The estimated price for the 2019 BMW with 45,000 miles is $21,383.

2. Significant price drivers:
   - **Age and Mileage**: The combination of the car's age and mileage is the most influential factor, indicating that lower mileage and newer vehicles typically command higher prices.
   - **Engine Power**: The horsepower of the engine also plays a significant role, with more powerful engines generally increasing the vehicle's value.
   - **Accident History**: The absence of any accidents and a clean title positively impact the car's value, as buyers prefer vehicles with a clean history.

3. It's important to note that this estimate is based on statistical